# Indexation FinanceBench avec BGE-M3 sur Kaggle

Ce notebook reprend l'indexation là où Colab s'est arrêté : les documents déjà dans Qdrant sont sautés.

**Réglages avant de lancer** (panneau de droite, **Settings** / **Session options**) :
1. **Accelerator** : `GPU T4 x2` (évitez P100).
2. **Internet** : `On` (nécessite un compte vérifié par téléphone).
3. Menu **Add-ons → Secrets** : ajoutez `QDRANT_URL` et `QDRANT_API_KEY` (et `GITHUB_TOKEN` si le dépôt est privé), puis cochez-les pour ce notebook.

## 1. Vérifier la carte graphique
Vous devez voir **Tesla T4**.

In [ ]:
!nvidia-smi

## 2. Récupérer le code du projet depuis GitHub

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
REPO = "HajarBraidi/financial-analyst"

try:
    token = secrets.get_secret("GITHUB_TOKEN")  # seulement si le dépôt est privé
    url = f"https://{token}@github.com/{REPO}.git"
except Exception:
    url = f"https://github.com/{REPO}.git"

%cd /kaggle/working
!rm -rf /kaggle/working/financial-analyst
!git clone -q {url} /kaggle/working/financial-analyst
%cd /kaggle/working/financial-analyst
!ls src

## 3. Installer les bibliothèques
PyTorch est déjà installé sur Kaggle. Les deux dernières lignes doivent afficher un numéro de version.

In [ ]:
!pip install -q -r requirements.txt
!pip show langchain-huggingface | grep Version
!pip show sentence-transformers | grep Version

## 4. Charger les clés Qdrant depuis les Secrets

In [ ]:
os.environ["QDRANT_URL"] = secrets.get_secret("QDRANT_URL")
os.environ["QDRANT_API_KEY"] = secrets.get_secret("QDRANT_API_KEY")
print("Clés chargées.")

## 5. Reprendre l'indexation
Vous devez voir `229 documents sélectionnés, XX déjà indexés, YY à faire.`

In [ ]:
!python src/indexing.py --tout

## 6. Vérifier la recherche (anglais et français)

In [ ]:
!python src/test_recherche.py